# Week 2-4 · 필수 티켓 pipeline과 평가표 만들기

## 시나리오
`classify → category-filtered retrieval → evidence gate → route`의 최소 pipeline을 직접 연결하고 여러 입력을 표 기반으로 평가합니다.

## 학습 목표
- 개별 단계의 입출력을 작은 dict로 연결한다.
- 근거가 없으면 planner 대신 `needs_more_information`으로 닫는다.
- category·citation·route 일관성을 여러 case로 검증한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · pipeline 함수

In [ ]:
# 실행 순서: 1단계 · pipeline 함수에서 classify_ticket, retrieve_playbook, evidence_gate을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · pipeline 함수.
practice_playbooks = {
    "billing": {"chunk_id": "bill-01", "text": "거래 ID를 확인합니다."},
    "access": {"chunk_id": "access-01", "text": "계정 잠금 상태를 확인합니다."},
}

# 텍스트를 category와 priority로 변환해 검색과 routing의 공통 입력으로 사용합니다.
def classify_ticket(text: str) -> dict:
    lowered = text.lower()
    if "invoice" in lowered or "charged" in lowered:
        return {"category": "billing", "priority": "normal"}
    if "login" in lowered:
        return {"category": "access", "priority": "normal"}
    if "outage" in lowered:
        return {"category": "technical", "priority": "urgent"}
    return {"category": "other", "priority": "normal"}

# 분류 category에 해당하는 playbook만 반환하고 없으면 빈 목록을 유지합니다.
def retrieve_playbook(category: str) -> list[dict]:
    document = practice_playbooks.get(category)
    return [document] if document else []

# citation에 필요한 필드가 있는 근거만 답변 단계로 통과시킵니다.
def evidence_gate(documents: list[dict]) -> bool:
    return bool(documents)

# urgent는 incident로 보내고 근거 없는 other는 canonical general queue로 보냅니다.
def choose_ticket_route(classification: dict) -> str:
    if classification["priority"] == "urgent":
        return "incident_escalation"
    if classification["category"] == "other":
        return "general_queue"
    return f'{classification["category"]}_queue'

# 분류 → 검색 → evidence gate → route 순서를 하나의 축소 pipeline으로 연결합니다.
def practice_ticket_pipeline(text: str) -> dict:
    classification = classify_ticket(text)
    documents = retrieve_playbook(classification["category"])
    if not evidence_gate(documents):
        return {**classification, "status": "needs_more_information", "route": choose_ticket_route(classification), "citations": []}
    return {**classification, "status": "plan_ready", "route": choose_ticket_route(classification), "citations": [documents[0]["chunk_id"]]}

### 2단계 · 평가 fixture 실행

In [ ]:
# 실행 순서: 2단계 · 평가 fixture 실행에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · 평가 fixture 실행.
evaluation_cases = [
    ("Duplicate invoice", "billing", "billing_queue", "plan_ready"),
    ("Cannot login", "access", "access_queue", "plan_ready"),
    ("Global outage", "technical", "incident_escalation", "needs_more_information"),
    ("Pricing question", "other", "general_queue", "needs_more_information"),
]
practice_results = [practice_ticket_pipeline(text) for text, *_ in evaluation_cases]
practice_results

### 3단계 · table-driven assertions

In [ ]:
# 실행 순서: 3단계 · table-driven assertions에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · table-driven assertions.
for result, (_, category, route, status) in zip(practice_results, evaluation_cases):
    assert result["category"] == category
    assert result["route"] == route
    assert result["status"] == status
    if status == "needs_more_information":
        assert result["citations"] == []
assert practice_results[0]["citations"] == ["bill-01"]
{"case_count": len(evaluation_cases), "passed": True}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
검색 근거가 없으면 해결 계획을 상상하지 않습니다. 평가 실패를 숨기거나 실제 고객 조치를 실행하지 않습니다.

## 실제 app 연결
Week 2 app을 이해하려면 분류 결과가 검색 filter와 route를 동시에 제어한다는 사실이 중요합니다. 그래서 이 Notebook은 라이브러리 한 개가 아니라 꼭 필요한 축소 pipeline을 실습합니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 Week 3의 `01_multi_agent_state_ownership.ipynb`에서는 여러 역할이 공유 state를 안전하게 나누어 쓰는 규칙을 학습합니다.